In [38]:
import pandas as pd
import pulp as pl

In [39]:
price_df = pd.DataFrame({
    ('GT', 50): [6], ('GT',100): [10],
    ('BT', 50): [7], ('BT',100): [12],
    ('WT', 50): [9], ('WT',100): [16],
    ('RT', 50): [8], ('RT',100): [14],
}, index=['price_USD'])

price_df

GT      BT      WT      RT    
          50  100 50  100 50  100 50  100
price_USD   6  10   7  12   9  16   8  14

In [40]:
raw_cost_100 = {'GT': 3.50, 'BT': 4.50, 'WT': 5.50, 'RT': 5.00}

raw_cost_df = pd.DataFrame({
    ('GT', 50): [raw_cost_100['GT'] * 0.5], ('GT',100): [raw_cost_100['GT']],
    ('BT', 50): [raw_cost_100['BT'] * 0.5], ('BT',100): [raw_cost_100['BT']],
    ('WT', 50): [raw_cost_100['WT'] * 0.5], ('WT',100): [raw_cost_100['WT']],
    ('RT', 50): [raw_cost_100['RT'] * 0.5], ('RT',100): [raw_cost_100['RT']],
}, index=['raw_cost_USD'])

raw_cost_df

GT         BT         WT        RT     
               50   100   50   100   50   100  50   100
raw_cost_USD  1.75  3.5  2.25  4.5  2.75  5.5  2.5  5.0

In [41]:
dist_cost_df = pd.DataFrame(
    {'dist_cost_per_box_USD': [0.50, 0.40, 0.60, 0.70, 0.80, 1.00]},
    index=['PT', 'ES', 'FR', 'IT', 'DE', 'PL']
)

dist_cost = dist_cost_df['dist_cost_per_box_USD']

dist_cost_df

,dist_cost_per_box_USD
PT,0.5
ES,0.4
FR,0.6
IT,0.7
DE,0.8
PL,1.0


In [42]:
demand_df = pd.DataFrame({
    ('GT', 50): {'PT': 10000,'ES': 15000,'FR': 20000,'IT': 13000,'DE': 18000,'PL': 12000},
    ('GT', 100): {'PT': 8000,'ES': 12000,'FR': 17000,'IT': 11000,'DE': 16000,'PL': 10000},
    ('BT', 50): {'PT': 9000,'ES': 13000,'FR': 18000,'IT': 12000,'DE': 17000,'PL': 11000},
    ('BT', 100): {'PT': 7000,'ES': 11000,'FR': 16000,'IT': 10000,'DE': 15000,'PL': 9000},
    ('WT', 50): {'PT': 5000,'ES': 8000,'FR': 10000,'IT': 7000,'DE': 9000,'PL': 6000},
    ('WT', 100): {'PT': 4000,'ES': 6000,'FR': 9000,'IT': 6000,'DE': 8000,'PL': 5000},
    ('RT', 50): {'PT': 6000,'ES': 10000,'FR': 12000,'IT': 9000,'DE': 11000,'PL': 8000},
    ('RT', 100): {'PT': 5000,'ES': 9000,'FR': 11000,'IT': 8000,'DE': 10000,'PL': 7000},
})

demand_df

GT            BT            WT           RT       
      50     100    50     100    50    100    50     100
PT  10000   8000   9000   7000   5000  4000   6000   5000
ES  15000  12000  13000  11000   8000  6000  10000   9000
FR  20000  17000  18000  16000  10000  9000  12000  11000
IT  13000  11000  12000  10000   7000  6000   9000   8000
DE  18000  16000  17000  15000   9000  8000  11000  10000
PL  12000  10000  11000   9000   6000  5000   8000   7000

In [43]:
PROD_CAP_BOXES = 2000000          
PROD_COST_PER_BOX = 1.0              
FIXED_MARKETING_COST = 150000.0         
FIXED_ANNUAL_COST = 1800000.0        
MIN_SHARE_PER_TYPE = 0.10            

raw_limit_grams = {
    'GT': 25000000,
    'BT': 30000000,
    'WT': 15000000,
    'RT': 20000000,
}

TEAS  = ['GT','BT','WT','RT']
SIZES = [50, 100]
CNTR  = ['PT','ES','FR','IT','DE','PL']

print("Constants loaded")

Constants loaded


In [44]:
x_df = demand_df.copy().astype(float)
x_df.head(6)

GT                BT                WT               RT         
        50       100      50       100      50      100      50       100
PT  10000.0   8000.0   9000.0   7000.0   5000.0  4000.0   6000.0   5000.0
ES  15000.0  12000.0  13000.0  11000.0   8000.0  6000.0  10000.0   9000.0
FR  20000.0  17000.0  18000.0  16000.0  10000.0  9000.0  12000.0  11000.0
IT  13000.0  11000.0  12000.0  10000.0   7000.0  6000.0   9000.0   8000.0
DE  18000.0  16000.0  17000.0  15000.0   9000.0  8000.0  11000.0  10000.0
PL  12000.0  10000.0  11000.0   9000.0   6000.0  5000.0   8000.0   7000.0

In [45]:
# Task 1: Calculate the total number of tea boxes distributed across all countries
total_boxes = int(x_df.sum().sum())
print("TASK 1 - Total number of tea boxes:", total_boxes)

TASK 1 - Total number of tea boxes: 499000


In [46]:
# TASK 2: Compute the total number of boxes distributed to each country
boxes_per_country_df = (
    x_df.sum(axis=1)              
       .astype(int)
       .rename('Total number of boxes')          
       .sort_index()
       .to_frame()               
)

boxes_per_country_df

,Total number of boxes
DE,104000
ES,84000
FR,113000
IT,76000
PL,68000
PT,54000


In [47]:
# TASK 3: Find the total number of boxes distributed for each tea type
boxes_per_type_df = (
    x_df.sum(axis=0)           
       .groupby(level=0).sum()  
       .astype(int)
       .rename('Boxes per tea type')
       .to_frame()
)

boxes_per_type_df

,Boxes per tea type
BT,148000
GT,162000
RT,106000
WT,83000


In [48]:
# TASK 4: Determine how many 50g and 100g boxes were distributed in total
boxes_per_size_df = (
    x_df.sum(axis=0)           
       .groupby(level=1).sum()
       .astype(int)
       .rename('Boxes per size')
       .to_frame()
)

boxes_per_size_df

,Boxes per size
50,269000
100,230000


In [49]:
# TASK 5: Calculate the total revenue
revenue = float(x_df.mul(price_df.loc['price_USD'], axis=1).sum().sum())

print("TASK 5 - Total revenue (USD):", round(revenue, 2))

TASK 5 - Total revenue (USD): 4805000.0


In [50]:
# TASK 6: Compute the operating income by breaking down each cost component

revenue = x_df.mul(price_df.loc['price_USD'], axis=1).sum().sum()

raw_cost = x_df.mul(raw_cost_df.loc['raw_cost_USD'], axis=1).sum().sum()

dist_cost_total = x_df.sum(axis=1).mul(dist_cost, axis=0).sum()

prod_cost_total = x_df.sum().sum() * PROD_COST_PER_BOX

fixed_total = FIXED_MARKETING_COST + FIXED_ANNUAL_COST

operating_income = revenue - raw_cost - dist_cost_total - prod_cost_total - fixed_total
print("TASK 6 - Operating income:", round(operating_income, 2))

TASK 6 - Operating income: 401450.0


In [51]:
# TASK 7: Calculate the total grams of raw tea used for each tea type and verify if it meets the availability constraints

grams_used = {}
for i in TEAS:
    grams_used[i] = int(50 * x_df[(i, 50)].sum() + 100 * x_df[(i, 100)].sum())

raw_usage_check = pd.DataFrame({
    'grams_used': pd.Series(grams_used),
    'limit_grams': pd.Series(raw_limit_grams)
})
raw_usage_check['ok'] = raw_usage_check['grams_used'] <= raw_usage_check['limit_grams']

print("TASK 7 - Raw tea usage vs. availability (grams):")
raw_usage_check

TASK 7 - Raw tea usage vs. availability (grams):


,grams_used,limit_grams,ok
GT,11800000,25000000,True
BT,10800000,30000000,True
WT,6050000,15000000,True
RT,7800000,20000000,True


In [52]:
# TASK 8: Check if each tea type meets the market presence constraint (at least 10% of total boxes)

total_all = x_df.sum().sum()
print("TASK 8 - Market presence by tea type (need >= 10%):\n")

for tea in TEAS:
    share_boxes = x_df[(tea, 50)].sum() + x_df[(tea, 100)].sum()
    pct = 0.0 if total_all == 0 else (share_boxes / total_all) * 100
    ok = share_boxes >= MIN_SHARE_PER_TYPE * total_all
    print(f"{tea}: {pct:.2f}%  ->  {'OK' if ok else 'NOT OK'}")

TASK 8 - Market presence by tea type (need >= 10%):

GT: 32.46%  ->  OK
BT: 29.66%  ->  OK
WT: 16.63%  ->  OK
RT: 21.24%  ->  OK


In [53]:
# TASK 9: Identify which constraints (demand, capacity, raw tea availability, market presence) are violated in a given distribution plan

capacity_viol = (total_boxes > PROD_CAP_BOXES)

demand_viol = (x_df > demand_df).any().any()

tea_availability_viol = not raw_usage_check['ok'].all()

needed_each = MIN_SHARE_PER_TYPE * total_boxes if total_boxes > 0 else 0
market_presence_viol = (boxes_per_type_df['Boxes per tea type'] < needed_each).any()

print("TASK 9 - Constraint status\n")

print("Capacity:", "violated" if capacity_viol else "satisfied")
print("Demand limits:", "violated" if demand_viol else "satisfied")
print("Raw tea availability:", "violated" if tea_availability_viol else "satisfied")
print("Market presence (>=10% each):", "violated" if market_presence_viol else "satisfied")

TASK 9 - Constraint status

Capacity: satisfied
Demand limits: satisfied
Raw tea availability: satisfied
Market presence (>=10% each): satisfied


In [54]:
# TASK 10: Optimal operating income and Optimal distribution plan

def unit_margin(tea, size, country):
    price = float(price_df.loc['price_USD', (tea, size)])
    rawc = float(raw_cost_df.loc['raw_cost_USD', (tea, size)])
    distc = float(dist_cost.loc[country])
    return price - rawc - distc - PROD_COST_PER_BOX

model = pl.LpProblem("Tea_Operating_Income_Max", pl.LpMaximize)
x = {(i,j,k): pl.LpVariable(f"x_{i}_{j}_{k}", lowBound=0) for i in TEAS for j in SIZES for k in CNTR}

model += pl.lpSum(unit_margin(i,j,k) * x[(i,j,k)] for i in TEAS for j in SIZES for k in CNTR)


for i in TEAS:
    for j in SIZES:
        for k in CNTR:
            model += x[(i,j,k)] <= demand_df.loc[k, (i,j)]
model += pl.lpSum(x.values()) <= PROD_CAP_BOXES
for i in TEAS:
    model += pl.lpSum(j * x[(i,j,k)] for j in SIZES for k in CNTR) <= raw_limit_grams[i]
Total = pl.LpVariable("Total_boxes", lowBound=0)
model += Total == pl.lpSum(x.values())
for i in TEAS:
    model += pl.lpSum(x[(i,j,k)] for j in SIZES for k in CNTR) >= MIN_SHARE_PER_TYPE * Total


_ = model.solve(pl.PULP_CBC_CMD(msg=False))
print("Solve status:", pl.LpStatus[model.status])


rows = []
for k in CNTR:
    for i in TEAS:
        for j in SIZES:
            val = x[(i,j,k)].value()
            rows.append({'country': k, 'tea': i, 'size': j, 'boxes': 0.0 if val is None else float(val)})

x_opt_df = (
    pd.DataFrame(rows)
      .pivot_table(index='country', columns=['tea','size'], values='boxes', aggfunc='sum', fill_value=0.0)
      .reindex(index=CNTR, columns=demand_df.columns)
      .astype(float)
)

rev = x_opt_df.mul(price_df.loc['price_USD'], axis=1).sum().sum()
rawc = x_opt_df.mul(raw_cost_df.loc['raw_cost_USD'], axis=1).sum().sum()
distc = x_opt_df.sum(axis=1).mul(dist_cost, axis=0).sum()
prod = x_opt_df.sum().sum() * PROD_COST_PER_BOX
fixed = FIXED_MARKETING_COST + FIXED_ANNUAL_COST
opinc = rev - rawc - distc - prod - fixed

print("Optimal operating income (USD):", round(opinc, 2))
print("\nOptimal distribution plan:")
display(x_opt_df.round(2))


Solve status: Optimal
Optimal operating income (USD): 401450.0

Optimal distribution plan:


GT                BT                WT               RT         
             50       100      50       100      50      100      50       100
country                                                                       
PT       10000.0   8000.0   9000.0   7000.0   5000.0  4000.0   6000.0   5000.0
ES       15000.0  12000.0  13000.0  11000.0   8000.0  6000.0  10000.0   9000.0
FR       20000.0  17000.0  18000.0  16000.0  10000.0  9000.0  12000.0  11000.0
IT       13000.0  11000.0  12000.0  10000.0   7000.0  6000.0   9000.0   8000.0
DE       18000.0  16000.0  17000.0  15000.0   9000.0  8000.0  11000.0  10000.0
PL       12000.0  10000.0  11000.0   9000.0   6000.0  5000.0   8000.0   7000.0